In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from fastprogress import master_bar, progress_bar
from torch.utils.data import random_split
from torchsummary import summary
import pandas as pd
import requests
import os
from tqdm import tqdm
from urllib.parse import urlparse
import time
import re
from collections import Counter
from PIL import Image
from io import BytesIO


batch_size = 256

import torchsummary


torch.manual_seed(42)

In [13]:
# DO NOT CHANGE
use_cuda = True
use_cuda = False if not use_cuda else torch.cuda.is_available()
device = torch.device('cuda:0' if use_cuda else 'cpu')
torch.cuda.get_device_name(device) if use_cuda else 'cpu'
print('Using device', device)

Using device cuda:0


In [14]:
print("Lade listings.csv...")
df = pd.read_csv("/kaggle/input/datasets/nilsmatthiessen/airbnb-listings/listings-2.csv")
df.head(1)


Lade listings.csv...


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,3176,https://www.airbnb.com/rooms/3176,20250923202926,2025-09-24,city scrape,Fabulous Flat in great Location,This beautiful first floor apartment is situa...,The neighbourhood is famous for its variety of...,https://a0.muscache.com/pictures/hosting/Hosti...,3718,...,4.7,4.92,4.61,First name and Last name: Nicolas Krotz <br/> ...,f,1,1,0,0,0.76


In [4]:
target_col = 'price'
df['price'] = df['price'].replace(r'[\$,]', '', regex=True).astype(float)
# Entferne ungültige Preise
df = df.dropna(subset=['price'])
df = df[(df['price'] > 10) & (df['price'] < 2000)]

# FEATURE SELECTION
# A. Wichtige Numerische Spalten
num_cols = [
    'accommodates', 'bedrooms', 'beds', 'bathrooms', 
    'review_scores_rating', 'number_of_reviews', 'reviews_per_month',
    'minimum_nights', 'maximum_nights',
    'calculated_host_listings_count'
]

# B. Wichtige Kategoriale Spalten (für One-Hot Encoding)
cat_cols = ['room_type', 'property_type']

# C. Spatial Spalten (für Derived Feature)
spatial_cols = ['latitude', 'longitude']

# Prüfen, welche Spalten tatsächlich existieren
valid_num_cols = [c for c in num_cols if c in df.columns]
valid_cat_cols = [c for c in cat_cols if c in df.columns]
valid_spatial_cols = [c for c in spatial_cols if c in df.columns]

print(f"\nGefundene Numerische Spalten: {valid_num_cols}")
print(f"Gefundene Kategoriale Spalten: {valid_cat_cols}")
print(f"Gefundene Spatial Spalten: {valid_spatial_cols}")

# FEATURE ENGINEERING (Derived Features) ---

# A. Distanz zum Zentrum (Spatial)
if 'latitude' in df.columns and 'longitude' in df.columns:
    # Entferne Zeilen ohne Koordinaten
    df = df.dropna(subset=['latitude', 'longitude'])
    
    # Berechne das "Zentrum" aller Wohnungen 
    # -- Kann man später auch noch mit anderen Zentrum machen
    center_lat = df['latitude'].mean()
    center_lon = df['longitude'].mean()
    
    # Euklidische Distanz berechnen
    df['dist_to_center'] = np.sqrt((df['latitude'] - center_lat)**2 + (df['longitude'] - center_lon)**2)
    # Füge es der Liste der numerischen Features hinzu
    valid_num_cols.append('dist_to_center')

# B. Preis pro Person (Derived Feature)
# ACHTUNG: Wir nutzen den Preis hier als INPUT Feature, nicht als Target!
# Das hilft dem Modell zu verstehen: "Wenn es für 8 Personen ist, aber nur 100€ kostet, ist es billig"
if 'accommodates' in df.columns:
    # Vermeide Division durch Null
    df['price_per_person'] = df['price'] / df['accommodates'].replace(0, 1)
    valid_num_cols.append('price_per_person')

# --- 5. DATA CLEANING & TRANSFORMATION ---
# A. Numerische Daten: Fülle fehlende Werte (NaNs)
for col in valid_num_cols:
    median_val = df[col].median()
    if pd.isna(median_val): median_val = 0
    df[col] = df[col].fillna(median_val)

# B. Kategoriale Daten: One-Hot Encoding (zu Fuß mit Pandas)
# Wir erstellen Dummy-Spalten (0 oder 1)
dummy_cols_list = []
for col in valid_cat_cols:
    dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
    df = pd.concat([df, dummies], axis=1)
    dummy_cols_list.extend(list(dummies.columns))

# --- 6. ZUSAMMENFÜHREN ZU X und y ---

# Sammle alle Spaltennamen für X (Features)
final_feature_cols = valid_num_cols + dummy_cols_list

# Erstelle X Matrix und y Vektor
X_raw = df[final_feature_cols].values
y = df[target_col].values

print(f"\nFinal Shape X: {X_raw.shape}")
print(f"Final Shape y: {y.shape}")

# --- 7. SPEICHERN ---
np.save('X_filtered.npy', X_raw)
np.save('y.npy', y)

print("Bereinigte und gefilterte Daten gespeichert.") 


Gefundene Numerische Spalten: ['accommodates', 'bedrooms', 'beds', 'bathrooms', 'review_scores_rating', 'number_of_reviews', 'reviews_per_month', 'minimum_nights', 'maximum_nights', 'calculated_host_listings_count']
Gefundene Kategoriale Spalten: ['room_type', 'property_type']
Gefundene Spatial Spalten: ['latitude', 'longitude']

Final Shape X: (9229, 78)
Final Shape y: (9229,)
Bereinigte und gefilterte Daten gespeichert.


In [5]:
class AirbnbDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [6]:
# ÄNDERUNG HIER: allow_pickle=True hinzufügen
X_raw = np.load('X_filtered.npy', allow_pickle=True)
y = np.load('y.npy', allow_pickle=True)

# Optional: Sicherstellen, dass X wirklich nur Zahlen sind (Float)
# Falls durch das Pickle doch Strings reingerutscht sind:
try:
    X_raw = X_raw.astype(np.float32)
except ValueError as e:
    print(f"Fehler beim Umwandeln in Float: {e}")
    print("Bitte überprüfe die X_filtered.npy Datei auf Text-Inhalte.")

# y sollte auch Float sein
y = y.astype(np.float32)
#Manuelles Splitting (70% Train, 15% Val, 15% Test)
num_samples = len(X_raw)
indices = np.arange(num_samples)
np.random.shuffle(indices)

# Grenzen setzen
train_split = int(0.7 * num_samples)
val_split = int(0.85 * num_samples)

# Indices aufteilen
train_indices = indices[:train_split]
val_indices = indices[train_split:val_split]
test_indices = indices[val_split:]

print(f"Anzahl Trainingsdaten: {len(train_indices)}")
print(f"Anzahl Validierungsdaten: {len(val_indices)}")
print(f"Anzahl Testdaten: {len(test_indices)}")

# Daten aufteilen
X_train_raw = X_raw[train_indices]
X_val_raw = X_raw[val_indices]
X_test_raw = X_raw[test_indices]

y_train = y[train_indices]
y_val = y[val_indices]
y_test = y[test_indices]

mean = X_train_raw.mean(axis=0)
std = X_train_raw.std(axis=0)
std[std == 0] = 1.0

# Skalierung
X_train_scaled = (X_train_raw - mean) / std
X_val_scaled = (X_val_raw - mean) / std
X_test_scaled = (X_test_raw - mean) / std

print("Scaling abgeschlossen.")

train_dataset = AirbnbDataset(X_train_scaled, y_train)
val_dataset = AirbnbDataset(X_val_scaled, y_val)
test_dataset = AirbnbDataset(X_test_scaled, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nDataLoader bereit!")
print(f"Train Batches: {len(train_loader)}")
print(f"Validation Batches: {len(val_loader)}")
print(f"Test Batches: {len(test_loader)}")

Anzahl Trainingsdaten: 6460
Anzahl Validierungsdaten: 1384
Anzahl Testdaten: 1385
Scaling abgeschlossen.

DataLoader bereit!
Train Batches: 26
Validation Batches: 6
Test Batches: 6


In [7]:
def train(dataloader, optimizer, model, loss_fn, device, master_bar):
    """Training function with MSE"""
    model.train()
    batch_losses = []
    
    # Wir sammeln die quadrierten Fehler, um am Ende den RMSE zu berechnen
    sum_squared_errors = 0
    total_samples = 0
    
    for inputs, targets in progress_bar(dataloader, parent=master_bar):
        
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        if targets.dim() > 1:
            targets = targets.squeeze(1)

        optimizer.zero_grad()
        predictions = model(inputs)
        
        if predictions.dim() > 1:
            predictions = predictions.squeeze(1)

        loss = loss_fn(predictions, targets) # Das ist bereits der MSE Loss
        
        loss.backward()
        optimizer.step()
        
        batch_losses.append(loss.item())
        
        # Für RMSE Berechnung: Summe der quadrierten Fehler
        error = predictions - targets
        sum_squared_errors += torch.sum(error ** 2).item()
        total_samples += targets.size(0)
    
    mean_loss = sum(batch_losses) / len(batch_losses)
    
    # RMSE berechnen: Wurzel aus dem Durchschnitt der quadrierten Fehler
    rmse = np.sqrt(sum_squared_errors / total_samples)
    
    return mean_loss, rmse


def validate(dataloader, model, loss_fn, device, master_bar):
    """Validation function with MSE"""
    model.eval()
    epoch_losses = []
    sum_squared_errors = 0
    total_samples = 0
    
    with torch.no_grad():
        for inputs, targets in progress_bar(dataloader, parent=master_bar):
            
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            if targets.dim() > 1:
                targets = targets.squeeze(1)

            predictions = model(inputs)
            
            if predictions.dim() > 1:
                predictions = predictions.squeeze(1)
            
            loss = loss_fn(predictions, targets)
            
            epoch_losses.append(loss.item())
            
            error = predictions - targets
            sum_squared_errors += torch.sum(error ** 2).item()
            total_samples += targets.size(0)
    
    mean_loss = sum(epoch_losses) / len(epoch_losses)
    rmse = np.sqrt(sum_squared_errors / total_samples)
    
    return mean_loss, rmse

In [8]:
class LinearRegression(torch.nn.Module):
    """
    Linear regression model inherits the torch.nn.Module
    which is the base class for all PyTorch modules.
    """
    def __init__(self, input_dim, output_dim):
        """ Initializes internal Module state. """
        super(LinearRegression, self).__init__()
        # TODO: define a linear layer for the model using torch.nn.Linear
        self.linear = torch.nn.Linear(input_dim, output_dim)


    def forward(self, x):
        """ Defines the computation performed at every call. """
        # TODO: flatten the input to a suitable size for the linear layer 
        # Hint: use x.reshape(...) and the input_dim variable
        
        # Deine Zeile:
        # x = x.reshape(x.shape[0], -1)
        
        # Bessere Alternative (Standard in PyTorch):
        # -1 bedeutet automatisch: "Rechne die zweite Dimension aus"
        x = x.view(x.size(0), -1)
        
        # TODO: run the flattened data through the linear layer and return the outputs
        outputs = self.linear(x)
        return outputs

In [9]:
def run_training(model, optimizer, loss_function, device, num_epochs, train_dataloader, val_dataloader):
    """ method to run the training procedure """
    
    train_losses = []
    val_losses = []
    train_rmses = [] 
    val_rmses = []
    
    mb = master_bar(range(num_epochs))
    print(f"Starte Training für {num_epochs} Epochen...")
    
    for epoch in mb:
        train_loss, train_rmse = train(
            dataloader=train_dataloader,
            optimizer=optimizer,
            model=model,
            loss_fn=loss_function,
            device=device,
            master_bar=mb
        )
        train_losses.append(train_loss)
        train_rmses.append(train_rmse)
        
        val_loss, val_rmse = validate(
            dataloader=val_dataloader,
            model=model,
            loss_fn=loss_function,
            device=device,
            master_bar=mb
        )
        val_losses.append(val_loss)
        val_rmses.append(val_rmse)
        
        mb.write(f"Epoch {epoch+1}/{num_epochs} | "
                 f"Train Loss (MSE): {train_loss:.4f} | "
                 f"Train RMSE:       {train_rmse:.2f}€ | "
                 f"Val Loss (MSE):   {val_loss:.4f} | "
                 f"Val RMSE:         {val_rmse:.2f}€")
    
    print("Training beendet.")
    
    return (np.array(train_losses), 
            np.array(val_losses),  
            np.array(train_rmses),   
            np.array(val_rmses))

In [16]:
sample_inputs, _ = next(iter(train_loader))
input_dim = sample_inputs.shape[1]

model_linear = LinearRegression(input_dim, 1)
model_linear = model_linear.to(device)

loss_function = torch.nn.MSELoss() # Mean Squared Error für Regression
optimizer = torch.optim.Adam(model_linear.parameters(), lr=0.1) 
num_epochs = 100

In [17]:
train_losses, val_losses, train_mse, val_mse = run_training(model_linear, optimizer, loss_function, device,num_epochs, train_loader, val_loader)    


Starte Training für 100 Epochen...


<div><p>Epoch 1/100 | Train Loss (MSE): 27193.1294 | Train RMSE:       165.64€ | Val Loss (MSE):   26341.1110 | Val RMSE:         161.41€</p><p>Epoch 2/100 | Train Loss (MSE): 25334.7481 | Train RMSE:       159.07€ | Val Loss (MSE):   24526.1195 | Val RMSE:         155.75€</p><p>Epoch 3/100 | Train Loss (MSE): 24009.2222 | Train RMSE:       153.79€ | Val Loss (MSE):   23041.6068 | Val RMSE:         150.99€</p><p>Epoch 4/100 | Train Loss (MSE): 22146.4162 | Train RMSE:       149.20€ | Val Loss (MSE):   21769.8691 | Val RMSE:         146.82€</p><p>Epoch 5/100 | Train Loss (MSE): 21042.9976 | Train RMSE:       145.28€ | Val Loss (MSE):   20692.5579 | Val RMSE:         143.23€</p><p>Epoch 6/100 | Train Loss (MSE): 20359.9790 | Train RMSE:       141.82€ | Val Loss (MSE):   19721.8275 | Val RMSE:         139.89€</p><p>Epoch 7/100 | Train Loss (MSE): 19222.1085 | Train RMSE:       138.59€ | Val Loss (MSE):   18867.0130 | Val RMSE:         136.82€</p><p>Epoch 8/100 | Train Loss (MSE): 18158.2086 | Train RMSE:       135.56€ | Val Loss (MSE):   18060.5435 | Val RMSE:         133.93€</p><p>Epoch 9/100 | Train Loss (MSE): 17461.6539 | Train RMSE:       132.79€ | Val Loss (MSE):   17321.5482 | Val RMSE:         131.18€</p><p>Epoch 10/100 | Train Loss (MSE): 16762.4914 | Train RMSE:       130.12€ | Val Loss (MSE):   16626.1086 | Val RMSE:         128.53€</p><p>Epoch 11/100 | Train Loss (MSE): 16122.0086 | Train RMSE:       127.56€ | Val Loss (MSE):   15940.3643 | Val RMSE:         125.88€</p><p>Epoch 12/100 | Train Loss (MSE): 16076.9201 | Train RMSE:       125.04€ | Val Loss (MSE):   15321.1413 | Val RMSE:         123.44€</p><p>Epoch 13/100 | Train Loss (MSE): 15351.1611 | Train RMSE:       122.59€ | Val Loss (MSE):   14707.0340 | Val RMSE:         120.95€</p><p>Epoch 14/100 | Train Loss (MSE): 14379.5541 | Train RMSE:       120.20€ | Val Loss (MSE):   14119.1587 | Val RMSE:         118.52€</p><p>Epoch 15/100 | Train Loss (MSE): 13740.7041 | Train RMSE:       117.88€ | Val Loss (MSE):   13554.4469 | Val RMSE:         116.14€</p><p>Epoch 16/100 | Train Loss (MSE): 13201.0415 | Train RMSE:       115.62€ | Val Loss (MSE):   13016.8776 | Val RMSE:         113.84€</p><p>Epoch 17/100 | Train Loss (MSE): 12729.7584 | Train RMSE:       113.42€ | Val Loss (MSE):   12520.5713 | Val RMSE:         111.63€</p><p>Epoch 18/100 | Train Loss (MSE): 12747.8888 | Train RMSE:       111.28€ | Val Loss (MSE):   12022.4489 | Val RMSE:         109.40€</p><p>Epoch 19/100 | Train Loss (MSE): 11799.4395 | Train RMSE:       109.17€ | Val Loss (MSE):   11548.5435 | Val RMSE:         107.23€</p><p>Epoch 20/100 | Train Loss (MSE): 11340.6306 | Train RMSE:       107.09€ | Val Loss (MSE):   11089.3778 | Val RMSE:         105.08€</p><p>Epoch 21/100 | Train Loss (MSE): 10934.9424 | Train RMSE:       105.09€ | Val Loss (MSE):   10655.9447 | Val RMSE:         103.01€</p><p>Epoch 22/100 | Train Loss (MSE): 10662.1252 | Train RMSE:       103.11€ | Val Loss (MSE):   10238.8592 | Val RMSE:         100.99€</p><p>Epoch 23/100 | Train Loss (MSE): 10137.2887 | Train RMSE:       101.20€ | Val Loss (MSE):   9832.3283 | Val RMSE:         98.95€</p><p>Epoch 24/100 | Train Loss (MSE): 9881.1530 | Train RMSE:       99.30€ | Val Loss (MSE):   9447.8285 | Val RMSE:         97.00€</p><p>Epoch 25/100 | Train Loss (MSE): 9402.7177 | Train RMSE:       97.44€ | Val Loss (MSE):   9088.0098 | Val RMSE:         95.12€</p><p>Epoch 26/100 | Train Loss (MSE): 9058.1529 | Train RMSE:       95.63€ | Val Loss (MSE):   8719.7848 | Val RMSE:         93.17€</p><p>Epoch 27/100 | Train Loss (MSE): 8732.1507 | Train RMSE:       93.91€ | Val Loss (MSE):   8389.6279 | Val RMSE:         91.39€</p><p>Epoch 28/100 | Train Loss (MSE): 8395.3340 | Train RMSE:       92.19€ | Val Loss (MSE):   8067.7776 | Val RMSE:         89.58€</p><p>Epoch 29/100 | Train Loss (MSE): 8163.1723 | Train RMSE:       90.51€ | Val Loss (MSE):   7764.3427 | Val RMSE:         87.89€</p><p>Epoch 30/100 | Train Loss (MSE): 7839.5669 | Train RMS

Training beendet.


In [ ]:
def test_learning_rates(learning_rates, model_class, input_dim, output_dim, 
                        train_loader, val_loader, device, epochs=20):
    """
    Testet verschiedene Learning Rates und plottet den Validation Loss.
    
    Args:
        learning_rates (list): Liste der zu testenden Lernraten (z.B. [0.1, 0.01, 0.001])
        model_class: Die Klasse des Modells (z.B. LinearRegression)
        input_dim: Anzahl der Input-Features
        output_dim: Anzahl der Output-Features (meist 1)
        train_loader: DataLoader für Training
        val_loader: DataLoader für Validation
        device: 'cuda' oder 'cpu'
        epochs: Wie viele Epochen pro Lernrate trainiert werden soll
    """
    
    val_losses_history = [] # Speichert den finalen Val Loss für jede LR
    
    print(f"Starte Hyperparameter-Test für {len(learning_rates)} Learning Rates...")
    
    for lr in learning_rates:
        print(f"\n--- Teste Learning Rate: {lr} ---")
        
        # 1. Neues, frisches Modell initialisieren (WICHTIG!)
        # Wir müssen ein neues Modell erstellen, damit die Gewichte des vorherigen Laufs
        # nicht das Ergebnis verfälschen.
        model = model_class(input_dim=input_dim, output_dim=output_dim).to(device)
        
        # 2. Optimizer mit aktueller Learning Rate erstellen
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = torch.nn.MSELoss()
        
        # 3. Kurz trainieren
        # Wir nutzen hier eine vereinfachte Schleife ohne den ganzen Output-Overhead
        model.train()
        for epoch in range(epochs):
            for inputs, targets in train_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                
                # Shapes korrigieren
                if targets.dim() > 1: targets = targets.squeeze(1)
                
                optimizer.zero_grad()
                predictions = model(inputs)
                if predictions.dim() > 1: predictions = predictions.squeeze(1)
                
                loss = loss_fn(predictions, targets)
                loss.backward()
                optimizer.step()
        
        # 4. Validieren (Loss auf Validation Set berechnen)
        model.eval()
        val_loss_sum = 0
        count = 0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                if targets.dim() > 1: targets = targets.squeeze(1)
                
                predictions = model(inputs)
                if predictions.dim() > 1: predictions = predictions.squeeze(1)
                
                loss = loss_fn(predictions, targets)
                val_loss_sum += loss.item()
                count += 1
        
        final_val_loss = val_loss_sum / count
        val_losses_history.append(final_val_loss)
        
        print(f"Ergebnis LR {lr}: Finaler Val Loss (MSE) = {final_val_loss:.4f}")

    # --- PLOTTEN ---
    plt.figure(figsize=(10, 6))
    plt.plot(learning_rates, val_losses_history, marker='o', linestyle='-', color='b')
    plt.xscale('log')  # Logarithmische Skala für die X-Achse (Standard für LR)
    plt.xlabel('Learning Rate (log scale)')
    plt.ylabel('Validation Loss (MSE)')
    plt.title('Einfluss der Learning Rate auf den Validation Loss')
    plt.grid(True, which="both", ls="-", alpha=0.5)
    
    # Den besten Punkt markieren
    best_idx = np.argmin(val_losses_history)
    best_lr = learning_rates[best_idx]
    best_loss = val_losses_history[best_idx]
    plt.scatter(best_lr, best_loss, color='red', s=100, zorder=5, label=f'Bester: {best_lr}')
    plt.legend()
    
    plt.show()
    
    return best_lr, best_loss

In [ ]:
# Definition der zu testenden Lernraten
lrs_to_test = [1,0.1, 0.01, 0.001, 0.0001, 0.00001]

# Aufruf der Funktion
best_lr, best_loss = test_learning_rates(
    learning_rates=lrs_to_test,
    model_class=LinearRegression,  # Deine Modellklasse
    input_dim=input_dim,    # Die Dimension, die wir vorher ausgelesen haben
    output_dim=1,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=100  # 20 Epochen reichen für den Vergleich
)

print(f"\nDie beste Learning Rate ist: {best_lr} mit einem Loss von {best_loss}")

In [ ]:
class MLPModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(MLPModel, self).__init__()
        
        # Layer 1: Input -> Hidden (64 Neuronen)
        self.layer1 = nn.Linear(input_dim, 64)
        self.relu1 = nn.ReLU()
        
        # Layer 2: Hidden -> Hidden (32 Neuronen)
        self.layer2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        
        # Layer 3: Hidden -> Output (1 Neuron)
        self.layer3 = nn.Linear(32, 1)
        
        # Dropout (optional): Schaltet während des Trainings zufällig Neuronen aus,
        # um Overfitting zu verhindern. 20% der Neuronen werden ausgeschaltet.
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # Daten durch Layer 1
        x = self.layer1(x)
        x = self.relu1(x)
        x = self.dropout(x)
        
        # Daten durch Layer 2
        x = self.layer2(x)
        x = self.relu2(x)
        x = self.dropout(x)
        
        # Daten durch Output Layer (Keine ReLU am Ende, da wir jeden Preis vorhersagen wollen!)
        x = self.layer3(x)
        return x

In [ ]:
model_mlp = MLPModel(input_dim=input_dim, output_dim=1).to(device)

# 2. Loss Function und Optimizer
loss_function = nn.MSELoss()
# Wir probieren 0.01, da MLPs empfindlicher auf große Sprünge reagieren können als lineare Modelle
optimizer = torch.optim.Adam(model_mlp.parameters(), lr=0.1) 


In [ ]:
# Training starten
train_losses, val_losses, train_rmses, val_rmses = run_training(
    model=model_mlp,
    optimizer=optimizer,
    loss_function=loss_function,
    device=device,
    num_epochs=100,  # Mehr Epochen für das komplexere Modell
    train_dataloader=train_loader,
    val_dataloader=val_loader
)

In [ ]:

# 1. Den Index des minimalen Validation RMSE finden
best_epoch_idx = np.argmin(val_rmses)

# 2. Die Werte an diesem Index auslesen
best_epoch = best_epoch_idx + 1 # Epochen fangen bei 1 an für Menschen
best_val_rmse = val_rmses[best_epoch_idx]
best_train_rmse = train_rmses[best_epoch_idx]
best_val_loss = val_losses[best_epoch_idx]

print("="*50)
print(f"BESTER DURCHLAUF (Epoche {best_epoch}):")
print("-" * 50)
print(f"Train RMSE: {best_train_rmse:.2f}€")
print(f"Val RMSE:   {best_val_rmse:.2f}€  <-- BESTES ERGEBNIS")
print(f"Val Loss:   {best_val_loss:.4f}")
print("="*50)

In [ ]:
# Wir nutzen die validate Funktion, aber diesmal mit dem test_loader
test_loss, test_rmse = validate(
    dataloader=test_loader,
    model=model_mlp,
    loss_fn=loss_function,
    device=device,
    master_bar=master_bar(range(1)) # Dummy bar für die Funktion
)

print("\n" + "="*50)
print("FINALE TEST EVALUATION")
print("="*50)
print(f"Test RMSE: {test_rmse:.2f}€")
print(f"Test Loss (MSE): {test_loss:.4f}")
print("="*50)

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
resnet = models.resnet18(pretrained=True)
resnet = nn.Sequential(*list(resnet.children())[:-1]) 
resnet.to(device)
resnet.eval() # Evaluation mode (kein Training nötig)


In [ ]:
def extract_features_from_urls(url_list):
    features_list = []
    for url in tqdm(url_list, desc="Extrahiere Bild-Features"):
        try:
            response = requests.get(url, timeout=5)
            img = Image.open(BytesIO(response.content)).convert('RGB')
            img_tensor = preprocess(img).unsqueeze(0).to(device)
            
            with torch.no_grad():
                features = resnet(img_tensor)
            
            features_list.append(features.flatten().cpu().numpy())
        except:
            # Falls Fehler: Null-Vektor
            features_list.append(np.zeros(512))
            
    return np.array(features_list)


In [ ]:
df_sample = df.head(1000).copy()
urls = df_sample['picture_url'].tolist()
print("Starte Bild-Download und Feature Extraktion...")
X_images = extract_features_from_urls(urls)

# Speichern (damit wir das nicht neu machen müssen)
np.save('X_images.npy', X_images)
print(f"Bild-Features gespeichert! Shape: {X_images.shape}")

In [ ]:
# DAS IST ALLES, WAS NOCH BRACHT WIRD:
class MultimodalDataset(Dataset):
    def __init__(self, X, y):
        # X enthält jetzt schon: [Zahlen, Bild-Vektor]
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    
    def __len__(self): return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
# 1. Targets (y) aus dem Sample
y_sample = df_sample['price'].values

# 2. Tabulare Daten laden (DEINE EXISTIERENDE DATEI)
print("Lade X_filtered.npy...")
X_tabular = np.load('X_filtered.npy', allow_pickle=True)
X_images = np.load('X_images.npy')

# Sicherstellen, dass X_tabular Float ist
X_tabular = X_tabular.astype(np.float32)

# WICHTIG: Wir nehmen nur die ersten N_SAMPLES Zeilen aus X_tabular
# damit sie zu den Bildern passen
X_tabular_sample = X_tabular[:1000]

# 3. Zusammenfügen (Concatenation)
X_combined = np.hstack((X_tabular_sample, X_images))

print(f"Multimodaler Shape: {X_combined.shape}")


# =============================================================================
# TEIL 4: SPLITTING & SCALING
# =============================================================================

# Splitting (70/15/15)
num_samples = len(X_combined)
indices = np.arange(num_samples)
np.random.shuffle(indices)

train_split = int(0.7 * num_samples)
val_split = int(0.85 * num_samples)

train_idx = indices[:train_split]
val_idx = indices[train_split:val_split]
test_idx = indices[val_split:]

X_train = X_combined[train_idx]
X_val = X_combined[val_idx]
X_test = X_combined[test_idx]

y_train = y_sample[train_idx]
y_val = y_sample[val_idx]
y_test = y_sample[test_idx]

# Scaling (Mean/Std von Trainingsdaten)
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1.0

X_train_s = (X_train - mean) / std
X_val_s = (X_val - mean) / std
X_test_s = (X_test - mean) / std

print("Scaling abgeschlossen.")


# =============================================================================
# TEIL 5: DATALOADER
# =============================================================================

class MultimodalDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

batch_size = 32
train_dataset = MultimodalDataset(X_train_s, y_train)
val_dataset = MultimodalDataset(X_val_s, y_val)
test_dataset = MultimodalDataset(X_test_s, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("DataLoader bereit.")



In [ ]:
def train(dataloader, optimizer, model, loss_fn, device, master_bar):
    """Training function für Regression (RMSE)"""
    model.train()
    batch_losses = []
    sum_squared_errors = 0
    total_samples = 0
    
    for inputs, targets in progress_bar(dataloader, parent=master_bar):
        
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        # Shapes korrigieren (Batch, 1) -> (Batch)
        if targets.dim() > 1:
            targets = targets.squeeze(1)

        optimizer.zero_grad()
        predictions = model(inputs)
        
        if predictions.dim() > 1:
            predictions = predictions.squeeze(1)

        loss = loss_fn(predictions, targets)
        loss.backward()
        optimizer.step()
        
        batch_losses.append(loss.item())
        
        # RMSE Berechnung vorbereiten
        error = predictions - targets
        sum_squared_errors += torch.sum(error ** 2).item()
        total_samples += targets.size(0)
    
    mean_loss = sum(batch_losses) / len(batch_losses)
    rmse = np.sqrt(sum_squared_errors / total_samples)
    
    return mean_loss, rmse


def validate(dataloader, model, loss_fn, device, master_bar):
    """Validation function für Regression (RMSE)"""
    model.eval()
    epoch_losses = []
    sum_squared_errors = 0
    total_samples = 0
    
    with torch.no_grad():
        for inputs, targets in progress_bar(dataloader, parent=master_bar):
            
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            if targets.dim() > 1:
                targets = targets.squeeze(1)

            predictions = model(inputs)
            
            if predictions.dim() > 1:
                predictions = predictions.squeeze(1)
            
            loss = loss_fn(predictions, targets)
            
            epoch_losses.append(loss.item())
            
            # RMSE Berechnung vorbereiten
            error = predictions - targets
            sum_squared_errors += torch.sum(error ** 2).item()
            total_samples += targets.size(0)
    
    mean_loss = sum(epoch_losses) / len(epoch_losses)
    rmse = np.sqrt(sum_squared_errors / total_samples)
    
    return mean_loss, rmse


def run_training_Pictures(model, optimizer, loss_function, device, num_epochs, train_dataloader, val_dataloader):
    """Hauptfunktion, die das Training steuert"""
    
    train_losses = []
    val_losses = []
    train_rmses = [] 
    val_rmses = []
    
    mb = master_bar(range(num_epochs))
    print(f"Starte Training für {num_epochs} Epochen...")
    
    for epoch in mb:
        
        # 1. Trainieren
        train_loss, train_rmse = train(
            dataloader=train_dataloader,
            optimizer=optimizer,
            model=model,
            loss_fn=loss_function,
            device=device,
            master_bar=mb
        )
        train_losses.append(train_loss)
        train_rmses.append(train_rmse)
        
        # 2. Validieren
        val_loss, val_rmse = validate(
            dataloader=val_dataloader,
            model=model,
            loss_fn=loss_function,
            device=device,
            master_bar=mb
        )
        val_losses.append(val_loss)
        val_rmses.append(val_rmse)
        
        # 3. Ausgabe
        mb.write(f"Epoch {epoch+1}/{num_epochs} | "
                 f"Train Loss (MSE): {train_loss:.4f} | "
                 f"Train RMSE:       {train_rmse:.2f}€ | "
                 f"Val Loss (MSE):   {val_loss:.4f} | "
                 f"Val RMSE:         {val_rmse:.2f}€")
    
    print("Training beendet.")
    
    return (np.array(train_losses), 
            np.array(val_losses),  
            np.array(train_rmses),   
            np.array(val_rmses))

In [ ]:
class MLPModel_Small(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(MLPModel_Small, self).__init__()
        
        # Layer 1: Weniger Neuronen (Reduzierung der Komplexität)
        self.layer1 = nn.Linear(input_dim, 32)
        self.relu1 = nn.ReLU()
        
        # Layer 2: Noch kleiner
        self.layer2 = nn.Linear(32, 16)
        self.relu2 = nn.ReLU()
        
        # Layer 3: Output
        self.layer3 = nn.Linear(16, 1)
        
        # STARKER DROPOUT: 50% der Verbindungen werden zufällig deaktiviert
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu1(x)
        x = self.dropout(x)
        
        x = self.layer2(x)
        x = self.relu2(x)
        x = self.dropout(x)
        
        x = self.layer3(x)
        return x


In [ ]:
sample_inputs, _ = next(iter(train_loader))
input_dim = sample_inputs.shape[1]
print(f"Input Dimension: {input_dim}")

model_MLP_small = MLPModel_Small(input_dim=input_dim, output_dim=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_function = nn.MSELoss()

In [ ]:
# ... (Modell, Optimizer, etc. sind definiert) ...

# Training starten
train_losses, val_losses, train_rmses, val_rmses = run_training_Pictures(
    model=model_MLP_small,
    optimizer=optimizer,
    loss_function=loss_function,
    device=device,
    num_epochs=100,
    train_dataloader=train_loader,
    val_dataloader=val_loader
)

# Besten Durchlauf finden
best_epoch_idx = np.argmin(val_rmses)
print(f"\nBester Durchlauf war Epoche {best_epoch_idx+1} mit Val RMSE: {val_rmses[best_epoch_idx]:.2f}€")